# Training a Machine Learning model on apples and bananas

In this notebook we are going to train a ML model to look at an image and decide if it is an **apple** or a **banana**?

The model looks at lots of example photos, makes guesses, checks if it was right, and slowly gets better. This is called machine learning.

**How to use this notebook:** there are grey boxes with code in them. To run a box, click on it and press the **play button** (or hold **Shift** and press **Enter**). Run the boxes **in order**, from top to bottom.

## Step 0 : Helper Functions

Run the box below once to get everything ready. We'll install the libraries to run our program and will be creating some functions that would be used later on. It looks complicated, but you don't need to understand or change them.

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
import os, warnings
import random
import gc
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import tensorflow as tf
import matplotlib.pyplot as plt
tf.get_logger().setLevel("ERROR")

DATA_FOLDER = "dataset"
PICTURE_SIZE = (96, 96)  # every picture is shrunk to this size

train_pictures = None
test_pictures  = None
group_names    = None
training_record = None
the_model      = None



def load_pictures():
    """Find all the photos in the train and test folders."""
    global train_pictures, test_pictures, group_names
    train_pictures = tf.keras.utils.image_dataset_from_directory(
        DATA_FOLDER + "/train", image_size=PICTURE_SIZE,
        batch_size=8, label_mode="binary", shuffle=True)
    test_pictures = tf.keras.utils.image_dataset_from_directory(
        DATA_FOLDER + "/test", image_size=PICTURE_SIZE,
        batch_size=8, label_mode="binary", shuffle=False)
    group_names = train_pictures.class_names
    print("\nAll loaded! The program found two groups of pictures:", group_names)


def display_pictures(num_images=6):
    """Show a few example photos so we can see what the model sees."""
    pics, labels = next(iter(train_pictures))
    num_images = min(num_images, len(pics))
    chosen = random.sample(range(len(pics)), num_images)     # pick random ones, no repeats
    columns = min(3, num_images)                             # up to 3 pictures per row
    rows = (num_images + columns - 1) // columns             # enough rows to fit them all
    plt.figure(figsize=(3 * columns, 3 * rows))
    for slot, i in enumerate(chosen):
        plt.subplot(rows, columns, slot + 1)
        plt.imshow(pics[i].numpy().astype("uint8"))
        plt.title(group_names[int(labels[i][0])])
        plt.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close()


def build_model():
    tf.keras.backend.clear_session()   # free the previous model first
    gc.collect()
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=PICTURE_SIZE + (3,)),
        tf.keras.layers.Rescaling(1.0 / 255),
        tf.keras.layers.Conv2D(16, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model


def train_model(epochs=10):
    """Train the model on images. 'epochs' = number of times it looks at all the photos."""
    global training_record, model
    print("Starting model training...\n")
    model = build_model()
    training_record = model.fit(
        train_pictures, validation_data=test_pictures, epochs=epochs, verbose=2)
    print("\nFinished training for ", epochs, "time(s).")


def show_training_graph():
    """Draw a graph of how good the model got after each round of practice."""
    h = training_record.history
    practised = [x * 100 for x in h["accuracy"]]
    new_pics  = [x * 100 for x in h["val_accuracy"]]
    rounds = range(1, len(practised) + 1)
    plt.figure(figsize=(9, 5))
    plt.plot(rounds, practised, "o-", linewidth=2, label="training images")
    plt.plot(rounds, new_pics,  "s-", linewidth=2, label="test images")
    plt.title("model accuracy b/w training & test images")
    plt.xlabel("epochs")
    plt.ylabel("accuracy")
    plt.ylim(0, 100)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()


def display_model_predictions(num_images=6):
    """Show some random NEW photos and what the model guesses for each one."""
    mixed = test_pictures.unbatch().shuffle(1000)        # mix apples and bananas together
    pics, labels = next(iter(mixed.batch(num_images)))  
    guesses = model.predict(pics, verbose=0)
    n = len(pics)                                       
    columns = min(3, n)                                  
    rows = (n + columns - 1) // columns                  
    plt.figure(figsize=(3 * columns, 3 * rows))
    for i in range(n):
        plt.subplot(rows, columns, i + 1)
        plt.imshow(pics[i].numpy().astype("uint8"))
        guess = group_names[1] if guesses[i][0] > 0.5 else group_names[0]
        truth = group_names[int(labels[i][0])]
        result = "Correct!" if guess == truth else "Wrong"
        plt.title(f"ML prediction: {guess}\n{result}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close()

def test_random_pictures(folder="random_images", num_images=6):
    """Show the model some odd pictures and see what it guesses."""
    file_names = [f for f in os.listdir(folder)
                  if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    random.shuffle(file_names)
    file_names = file_names[:num_images]

    # load each picture and shrink it to the size the model expects
    pics = []
    for name in file_names:
        img = tf.keras.utils.load_img(os.path.join(folder, name), target_size=PICTURE_SIZE)
        pics.append(tf.keras.utils.img_to_array(img))
    pics = tf.convert_to_tensor(pics)

    guesses = model.predict(pics, verbose=0)

    n = len(pics)
    columns = min(3, n)
    rows = (n + columns - 1) // columns
    plt.figure(figsize=(3 * columns, 3 * rows))
    for i in range(n):
        plt.subplot(rows, columns, i + 1)
        plt.imshow(pics[i].numpy().astype("uint8"))
        chance_ = guesses[i][0]
        guess = group_names[1] if chance_ > 0.5 else group_names[0]
        sure = max(chance_, 1 - chance_) * 100
        plt.title(f"Model predicts: {guess}\n({sure:.0f}% sure)")
        plt.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close()

print("Setup complete")

## Step 1 : Load the pictures

This tells the program to go and find all your photos.

In [ ]:
load_pictures()

## Step 2 : Have a look at the pictures

Let's see a few of the photos, with their correct labels, so we know what the model is working with.

In [ ]:
# you can change the num_images parameter to different values to show more/different pictures
display_pictures(num_images=6)

## Step 3 : Train the model

The model will **learn**: it looks at the photos, makes guesses, finds out which it got wrong, and adjusts itself to do better next time.

![image](image.png)

One **epoch** means *"look at every single training photo once."* So `epochs=10` means it practises with the whole set of photos **10 times**.

### What to watch while it trains
After each round you'll see two numbers:

Accuracy : how often it gets it right. Higher is better (1.0 is perfect).
Loss : how wrong it is. Lower is better (0 is perfect).

Learning is going well when accuracy goes up and loss goes down.

Try `epochs=5` first. Later you can come back and try a bigger number like `20` or `30` to see what happens.

In [ ]:
train_model(epochs=10)

## Step 4 : Look at the learning graph

This graph shows how model is performing got after each epoch.

- **blue line** - how often it's right on the photos it **practised with**.
- **orange line** - how often it's right on **new** photos it has **never seen before**.

**Watch for this:** sometimes the blue line keeps climbing while the orange line stops rising (or even drops). That means the model is just **memorising** the training photos instead of
truly learning. Ideally, we would like to stop the training at a sweet spot where the orange line stops rising.

In [ ]:
show_training_graph()

## Step 5 : Test out some images

Let's show the model some new images and see what it guesses for each one.

In [ ]:
display_model_predictions(num_images=10)

## Try to trick the model
There are some pictures in random_images folder that are not apples or bananas (an orange, an aubergine...), see what the model predicts.
The model only knows two labels: apple and banana. So it has to pick one, can you think of why it would predict one over the other?




In [ ]:
test_random_pictures()

## Things to try

Go back to **Step 3**, change the number of epochs, and run all the cells again! You can do so using 'Run All' option at top of notebook.

1. With very few epochs (like `epochs=2`), how is the model performing on test images? 
2. What number of epochs makes the **orange line** (new photos) the highest?
3. You can add more pictures to the 'random_images' folder and see what model predicts!